##### Library imports

In [ ]:
import re, os, sys
import pandas as pd
import json
import cv2 as cv
import SimpleITK as sitk
import numpy as np 
from skimage import measure, filters
import matplotlib.pyplot as plt

##### Function definitions

In [ ]:
def window_stack_sitk(input_im, window_center=40, window_width=80):
    """
    Clamps values outside [min, max] to the edge values.
    
    Parameters
    ----------
    input_im : sitk.Image
        Input, unwindowed, brain image
    window_center : int, optional
        The center of the Hounsfield Unit (HU) scale in the windowed image. Default is 40 HU (brain).
    window_width : int, optional
        The total HU window width around the window_center, in the windowed image. Default is 80 HU (brain).

    Returns
    -------
    windowed_im : sitk.Image
        Windowed CT image with the desired tissue's visualization optimized (e.g., 0 - 80 HU for brain tissue)
    
    """
    img_min = window_center - (window_width // 2)
    img_max = window_center + (window_width // 2)


    windowed_im = sitk.IntensityWindowing(
        input_im, 
        windowMinimum=float(img_min), 
        windowMaximum=float(img_max), 
        outputMinimum=float(img_min), 
        outputMaximum=float(img_max)
    )
    
    return windowed_im

In [ ]:
def getLargestCC(blobs_labels):
    """Returns the largest connected component from an image containing blobs of discrete labels
    
    Parameters
    ----------
    blobs_labels : numpy.ndarray 
        A collection of discrete blobs (e.g., cerebrospinal fluid [CSF] spaces including the lateral ventricles, longitudinal fissure, sulci etc.)

    Returns
    -------
    largestCC: numpy.ndarray
        The largest connected component from a collection of discrete blobs (e.g. lateral ventricles from a collection of CSF spaces)  
    
    """
    # assume at least 1 CC apart from background
    if blobs_labels.max() == 0:
        raise ValueError('Blank segmentation, inspect processing up to here.')
    #this assumes that the background is the largest CC (label 0)
    largestCC = blobs_labels == np.argmax(np.bincount(blobs_labels.flat)[1:])+1 
    return largestCC

In [ ]:
def rotate_image(image, dimension = 3):
    """Rotates NIfTI axial images so they are upright for visualization. Simple insights tool kit (SITK) utilizes the LPS coordinate system, 
       meaning image voxel coordinates are assumed to increase in the Right --> Left, Anterior --> Posterior, and Inferior --> Superior directions. 
       However, NIfTI images use the RAS coordinate system, which makes axial slices (lateral cross-sections) appear inverted when read with SITK. 
       This function resamples the image such that it is visualized in upright direction, enabling interpretable visualization and processing. 
    
    Parameters
    ----------
    image : sitk.Image
        The original NIfTI image (Note that the direction cosine should be [-1,0,0,0,-1,0,0,0,1])
    dimension : int, optional
        Dimensionality of the original and rotation-corrected image. Default is 3.
    
    Returns
    -------
    rotated_image : sitk.Image
        Rotation-corrected image, with a direction cosine of [1,0,0,0,1,0,0,0,1]
    """

    transform = sitk.AffineTransform(dimension)
    transform.SetCenter(image.TransformContinuousIndexToPhysicalPoint(np.array(image.GetSize())//2.0))
    s = image.GetSpacing()[2]

    matrix = np.array([[1.0,0.0,0.0],[0.0,1.0,0.0],[0.0,0.0,1.0]])

   
    transform.SetMatrix(matrix.ravel())

    extreme_points = [image.TransformIndexToPhysicalPoint((0,0,0)), 
                      image.TransformIndexToPhysicalPoint((image.GetWidth(),0,0)),
                      image.TransformIndexToPhysicalPoint((image.GetWidth(),image.GetHeight(),0)),
                      image.TransformIndexToPhysicalPoint((0,image.GetHeight(),0)),
                      image.TransformIndexToPhysicalPoint((0,0,image.GetDepth())), 
                      image.TransformIndexToPhysicalPoint((image.GetWidth(),0,image.GetDepth())),
                      image.TransformIndexToPhysicalPoint((image.GetWidth(),image.GetHeight(),image.GetDepth())),
                      image.TransformIndexToPhysicalPoint((0,image.GetHeight(),image.GetDepth()))]

    inv_transform = transform.GetInverse()

    extreme_points_transformed = [inv_transform.TransformPoint(pnt) for pnt in extreme_points]
    min_x = min(extreme_points_transformed)[0]
    min_y = min(extreme_points_transformed, key=lambda p: p[1])[1]
    min_z = min(extreme_points_transformed, key=lambda p: p[2])[2]
    max_x = max(extreme_points_transformed)[0]
    max_y = max(extreme_points_transformed, key=lambda p: p[1])[1]
    max_z = max(extreme_points_transformed, key=lambda p: p[2])[2]

    # Use the original spacing (arbitrary decision).
    output_spacing = image.GetSpacing()
    # Identity cosine matrix.   
    output_direction = [1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0]
    # Minimal x,y coordinates are the new origin.
    output_origin = [min_x, min_y, min_z]
    
    # Compute grid size based on the physical size and spacing.
    output_size = image.GetSize()
    
    rotated_image = sitk.Resample(image, output_size, transform, sitk.sitkLinear, output_origin, output_spacing,
                                  output_direction)
    return rotated_image

In [ ]:
def translate_image(image, dimension, ac):
    """
    Translates the physical origin of the image so that the Anterior 
    Commissure (AC) world coordinates are set to (0, 0, 0).
    
    Parameters
    ----------
    image : sitk.Image
        The 3D SimpleITK image to be translated.
    dimension : int
        The dimensionality of the image. (Note: currently unused in the 
        function logic, as the transform is hardcoded to 3D).
    ac : tuple or list of float
        The (x, y, z) world coordinates of the Anterior Commissure.

    Returns
    -------
    translated_image : sitk.Image
        The resampled SimpleITK image with the translated spatial domain.
    """
    transform = sitk.AffineTransform(3)
    transform.SetCenter(image.TransformContinuousIndexToPhysicalPoint(np.array(image.GetSize())//2.0))
    transform.SetTranslation([x for x in ac])
    matrix = np.array([[1.0,0.0,0.0],[0.0,1.0,0.0],[0.0,0.0,1.0]])
    transform.SetMatrix(matrix.ravel())
    extreme_points = [image.TransformIndexToPhysicalPoint((0,0,0)), 
                      image.TransformIndexToPhysicalPoint((image.GetWidth(),0,0)),
                      image.TransformIndexToPhysicalPoint((image.GetWidth(),image.GetHeight(),0)),
                      image.TransformIndexToPhysicalPoint((0,image.GetHeight(),0)),
                      image.TransformIndexToPhysicalPoint((0,0,image.GetDepth())), 
                      image.TransformIndexToPhysicalPoint((image.GetWidth(),0,image.GetDepth())),
                      image.TransformIndexToPhysicalPoint((image.GetWidth(),image.GetHeight(),image.GetDepth())),
                      image.TransformIndexToPhysicalPoint((0,image.GetHeight(),image.GetDepth()))]
    inv_transform = transform.GetInverse()
    extreme_points_transformed = [inv_transform.TransformPoint(pnt) for pnt in extreme_points]
    min_x = min(extreme_points_transformed)[0]
    min_y = min(extreme_points_transformed, key=lambda p: p[1])[1]
    min_z = min(extreme_points_transformed, key=lambda p: p[2])[2]
    max_x = max(extreme_points_transformed)[0]
    max_y = max(extreme_points_transformed, key=lambda p: p[1])[1]
    max_z = max(extreme_points_transformed, key=lambda p: p[2])[2]
    # Use the original spacing (arbitrary decision).
    output_spacing = image.GetSpacing()
    # Identity cosine matrix.   
    output_direction = [1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0]
    # Minimal x,y coordinates are the new origin.
    output_origin = [min_x, min_y, min_z]

    # Compute grid size based on the physical size and spacing.
    output_size = image.GetSize()

    translated_image = sitk.Resample(image, output_size, transform, sitk.sitkLinear,output_origin,output_spacing,
                                      output_direction)
    return translated_image

In [ ]:
def ellipse_mask(cen, a, b, shape):
    """
    Generates a 2D binary mask containing an ellipse.
    
    Parameters
    ----------
    cen : tuple or list of int
        The (row, column) coordinates defining the center of the ellipse.
    a : float or int
        The semi-axis length along the row dimension (vertical).
    b : float or int
        The semi-axis length along the column dimension (horizontal).
    shape : tuple of int
        The (height, width) dimensions of the desired output mask array.

    Returns
    -------
    mask : numpy.ndarray
        A 2D binary array of the specified shape. Pixels falling inside or 
        on the boundary of the ellipse are set to 1, and all others are 0.
    """
    mask = np.zeros(shape)
    for row in range(shape[0]):
        for col in range(shape[1]):
            if (row - int(cen[0]))**2/a**2 + (col - int(cen[1]))**2/b**2 <= 1:
                mask[row, col] = 1
    return mask

In [ ]:
def ei_calc(axial_data, ac_indices, ei_calc_processing_params, visualize = False):

    """
    Calculates the Evans Index-x (EI-x) from axial brain CT data.

    This function iterates through axial slices superior to the anterior commissure (AC) 
    to locate the maximal width of the frontal horns of the lateral ventricles. It then 
    calculates the Evans Index by dividing this maximal ventricular width by the 
    maximal internal diameter of the skull at the same slice level.

    Parameters
    ----------
    axial_data : numpy.ndarray
        A 3D array representing the axial CT volume.
    ac_indices : list or tuple of int
        The 3D coordinates [x, y, z] of the anterior commissure. The z-coordinate 
        (index 2) is used as the starting slice for the evaluation.
    ei_calc_processing_params : dict
        A dictionary containing predefined, emperically derived parameters for thresholding,
        morphological operations, and Region of Interest (ROI) definitions. 
    visualize : bool, optional
        If True, displays frontal horn lengths across the evaluated slices. Default is False.

    Returns
    -------
    ei : float or None
        The calculated Evans Index. Returns None if a valid measurement could 
        not be extracted from the provided slices.
    plane_opt : int or None
        The axial slice index (z-coordinate) where the maximum frontal horn width 
        was successfully found. Returns None if extraction fails.
    """

    ei_x_num_slices_above_ac_to_eval = ei_calc_processing_params["ei_x_num_slices_above_ac_to_eval"]
    around_lv_roi_col_left_prcntl_lim = ei_calc_processing_params["around_lv_roi_col_left_prcntl_lim"]
    around_lv_roi_col_right_prcntl_lim = ei_calc_processing_params["around_lv_roi_col_right_prcntl_lim"]
    around_lv_roi_row_upper_prcntl_lim = ei_calc_processing_params["around_lv_roi_row_upper_prcntl_lim"]
    around_lv_roi_row_lower_prcntl_lim = ei_calc_processing_params["around_lv_roi_row_lower_prcntl_lim"]
    around_lv_roi_disk_size = ei_calc_processing_params["around_lv_roi_disk_size"]
    axial_adaptive_thresh_size = ei_calc_processing_params["axial_adaptive_thresh_size"]
    axial_adaptive_thresh_mean_thresh = ei_calc_processing_params["axial_adaptive_thresh_mean_thresh"]
    otsu_adaptive_diff_thresh = ei_calc_processing_params["otsu_adaptive_diff_thresh"]
    otsu_adaptive_horizontal_thresh = ei_calc_processing_params["otsu_adaptive_horizontal_thresh"]
    lv_h_roi_col_thresh_prcnt = ei_calc_processing_params["lv_h_roi_col_thresh_prcnt"]
    lv_h_roi_top_row_lim = ei_calc_processing_params["lv_h_roi_top_row_lim"]
    lv_h_roi_bot_row_lim = ei_calc_processing_params["lv_h_roi_bot_row_lim"]

    start_slice = ac_indices[2] 

    max_fh_width = 0
    fh_width_list = []

    axial_brain_im_opt = None
    full_brain_mask_opt = None
    plane_opt = None
    lv_x_opt, lv_y_opt = 0, 0
    rv_x_opt, rv_y_opt = 0, 0
    
    for plane in range(start_slice, start_slice+ei_x_num_slices_above_ac_to_eval):

        if plane >= axial_data.shape[0]:
            break
            
        axial_brain_im = axial_data[plane,:,:]      
        axial_uint8 = np.uint8(axial_brain_im)
        thresh = filters.threshold_otsu(axial_uint8)
        axial_bin = np.where(axial_uint8 > thresh,1,0)      

        M0 = cv.moments(np.uint8(axial_bin))
        if M0["m00"] == 0: 
            fh_width_list.append(0)
            continue
            
        cX = int(M0["m10"] / M0["m00"])
        cY = int(M0["m01"] / M0["m00"])
        [a_r,a_c] = axial_bin.nonzero()

        if len(a_c) == 0 or len(a_r) == 0:
            fh_width_list.append(0)
            continue
            
        # we want this ellipse to be smaller in row range so it fits well within the brain
        otsu_bin_col_range = np.percentile(a_c,around_lv_roi_col_right_prcntl_lim)-np.percentile(a_c,around_lv_roi_col_left_prcntl_lim)
        otsu_bin_row_range = np.percentile(a_r,around_lv_roi_row_lower_prcntl_lim) - np.percentile(a_r,around_lv_roi_row_upper_prcntl_lim)            
        
        a = otsu_bin_col_range//2
        b = otsu_bin_row_range//2
        if otsu_bin_row_range > otsu_bin_col_range:
            a, b = b, a   
            
        around_ventricle_mask = (ellipse_mask((cY,cX),a,b,axial_bin.shape) - 
                                ellipse_mask((cY,cX),a-around_lv_roi_disk_size,b-around_lv_roi_disk_size,axial_bin.shape))


        mask_for_full_brain = np.logical_or(axial_bin,around_ventricle_mask)
        
        #Floodfilling is used to extract the entire brain mask 
        im_th = mask_for_full_brain.copy()
        h, w = im_th.shape[:2]
        mask = np.zeros((h+2, w+2), np.uint8)
        im_floodfill = im_th.copy()
        # Floodfill from point (0, 0)
        res = cv.floodFill(np.uint8(im_floodfill), mask, (0,0), 255)
        full_brain_mask = (1-res[2]).copy()
        full_brain_mask = full_brain_mask[1:h+1,1:w+1]
        M = cv.moments(np.uint8(full_brain_mask))
        cX = int(M["m10"] / M["m00"])
        cY = int(M["m01"] / M["m00"])



        axial_ad_th = cv.adaptiveThreshold(axial_uint8,1,cv.ADAPTIVE_THRESH_MEAN_C,
                                            cv.THRESH_BINARY,axial_adaptive_thresh_size,axial_adaptive_thresh_mean_thresh) 
        
        background_pixels = (1-full_brain_mask)
        axial_brain_mask = np.where(background_pixels>0,0,axial_ad_th)


        adaptive_ventricles = np.where(full_brain_mask - axial_brain_mask>0,1,0)
        tm = np.zeros(adaptive_ventricles.shape)
        tm[:adaptive_ventricles.shape[0]//2,:]=1

        diff_otsu_adaptive = np.zeros_like(adaptive_ventricles)

        try:
            otsu_ventricles = getLargestCC(measure.label(np.where(full_brain_mask - axial_bin>0,1,0),background = 0))
            diff_otsu_adaptive = np.where(otsu_ventricles-adaptive_ventricles>0,1,0)*tm
            
            if (np.sum(diff_otsu_adaptive) > 0) and (np.sum((adaptive_ventricles)*tm) > 0):
                Ma = cv.moments(np.uint8(adaptive_ventricles)*tm)
                aX = int(Ma["m10"] / Ma["m00"]) if Ma["m00"] != 0 else 0
                aY = int(Ma["m01"] / Ma["m00"]) if Ma["m00"] != 0 else 0
                
                diff_density = np.sum(diff_otsu_adaptive)/np.sum(otsu_ventricles*tm)
                Md = cv.moments(np.uint8(diff_otsu_adaptive))
                dX = int(Md["m10"] / Md["m00"]) if Md["m00"] != 0 else 0
                dY = int(Md["m01"] / Md["m00"]) if Md["m00"] != 0 else 0

                diff_check = (diff_density > otsu_adaptive_diff_thresh) and (np.abs(aX-dX) < otsu_adaptive_horizontal_thresh)

                if diff_check:
                    ventricles = np.logical_or(adaptive_ventricles, diff_otsu_adaptive)
                else: ventricles = adaptive_ventricles
            else: ventricles = adaptive_ventricles
        except (ValueError, IndexError, ZeroDivisionError): 
            ventricles = adaptive_ventricles


        if np.sum(ventricles)==0:
            fh_width_list = fh_width_list + [0]
            continue

        h_roi = np.zeros(ventricles.shape)
        [r_v, c_v] = ventricles.nonzero()
        [fr, fc] = full_brain_mask.nonzero()
        if len(fc) == 0:
            fh_width_list.append(0)
            continue
            
        fbc_range = max(fc) - min(fc)

        col_thresh = int(fbc_range*lv_h_roi_col_thresh_prcnt)
        
        
        r_start = max(0, cY - lv_h_roi_top_row_lim)
        r_end = max(0, cY - lv_h_roi_bot_row_lim)
        c_start = max(0, cX - col_thresh)
        c_end = min(ventricles.shape[1], cX + col_thresh)
        h_roi[r_start:r_end, c_start:c_end] = 1
        
        ventricles_uh = ventricles.copy()
        ventricles_uh[cY:, :] = 0
        ventricles_connected = np.logical_or(ventricles_uh, h_roi)


        #This step extracts any remaining background pixels and eliminates them 
        try:
            [r,c] = (1-getLargestCC(measure.label(ventricles_connected, background = 0))).nonzero() #here
            ventricles_fh = ventricles_uh.copy() #here
            ventricles_fh[r,c] = 0
        except: 
            fh_width_list = fh_width_list + [0]
            continue

        #This gets rid of the lower part of the lateral ventricles
        fh_roi = np.ones(ventricles_fh.shape)
        fh_roi[ac_indices[1]:,:]=0
        ventricles_connected_fh = fh_roi*ventricles_fh

        [v_row, v_col] = ventricles_connected_fh.nonzero()
        if len(v_row) == 0:
            fh_width_list = fh_width_list + [0]
            continue 
        top_row = min(v_row)
        top_cols = sorted(v_col[v_row == top_row])
        max_dist = top_cols[-1] - top_cols[0]
        lv_x, lv_y = top_cols[0], top_row
        rv_x, rv_y = top_cols[-1], top_row

        for rows in np.unique(sorted(v_row)):
            top_cols = sorted(v_col[v_row == rows])
            dist = top_cols[-1] - top_cols[0]
            if dist > max_dist:
                max_dist = dist
                lv_x, lv_y = top_cols[0], rows
                rv_x, rv_y = top_cols[-1], rows


        fh_width = rv_x - lv_x
        fh_width_list = fh_width_list + [fh_width]
        if fh_width > max_fh_width:
            max_fh_width = fh_width 
            axial_brain_im_opt = axial_brain_im 
            full_brain_mask_opt = full_brain_mask #new addition
            plane_opt = plane
            lv_x_opt, lv_y_opt = lv_x, lv_y
            rv_x_opt, rv_y_opt = rv_x, rv_y
            
    if max_fh_width == 0 or axial_brain_im_opt is None:
        print("Warning: Could not isolate frontal horns to calculate EI-x.")
        return None, None
    
    row, col = axial_brain_im_opt[axial_brain_im_opt.shape[0]//2:, :].nonzero()
    if len(col) == 0:
        return None, None
        
    brain_l_x = min(col)
    brain_l_y = max(row[col == brain_l_x]) + axial_brain_im_opt.shape[0]//2
    brain_r_y = brain_l_y 
    brain_r_x = max(col[row == max(row[col == brain_l_x])]) 

    fig, ax = plt.subplots(figsize = (10,10))
    ax.imshow(axial_brain_im_opt, cmap = 'gray')
    ax.scatter([x for x in range(lv_x_opt,rv_x_opt+1)], [lv_y_opt]*(rv_x_opt - lv_x_opt+1),  marker = 'x',s = 4, 
           color = "w")
    ax.scatter([x for x in range(brain_l_x,brain_r_x+1)], [brain_l_y]*(brain_r_x - brain_l_x+1),  marker = 'x',s = 4,
           color = "w")
    plt.show()

    if visualize:
        plt.plot([x for x in range(len(fh_width_list))], fh_width_list)
        plt.scatter(plane_opt-(start_slice), fh_width_list[plane_opt-(start_slice+20)],marker = 'x')
        plt.title('Variation of FH length')
        plt.show()

    brain_width = brain_r_x - brain_l_x
    ei = (max_fh_width) / brain_width if brain_width > 0 else -100
    
    return ei, plane_opt

##### Data setup

###### Before running the pipeline, make a copy of config.template.json, rename it to config.json, and update the paths to point to your local dataset.

In [ ]:
# Load the configuration file
with open('config.json', 'r') as file:
    config = json.load(file)

# Assign variables dynamically
data_path = config['data_path']
axial_acpc_vol_aligned_name = config['axial_acpc_vol_aligned_name']
info_csv_path = config['info_csv_path_w_name']
scan_id_col_name = config['scan_id_col_name']
pat_id_col_name = config['pat_id_col_name']
ac_coordinates_col_name = config['ac_coordinates_col_name']
pc_coordinates_col_name = config['pc_coordinates_col_name']

output_folder = config['output_csv_write_folder']

print(f"Loading data from: {data_path}")

In [ ]:
info_df = pd.read_csv(info_csv_path)
info_df[scan_id_col_name] = info_df[scan_id_col_name].str.strip("'")

In [ ]:
accnum_list = info_df[scan_id_col_name].values

In [ ]:
len(accnum_list), len(info_df)

In [ ]:
#dataframe which will contain track of the computed EI-x values for each scan
ei_x_df = pd.DataFrame()

###### Predefined, emperically determined processing parameters corresponding to ROI sizes, positions, and global/local thresholding parameters. 
The following parameters (ei_calc_processing_params) work for a diverse cohort of chronic neurodegenerative conditions spanning Normal Pressure Hydrocephalus, Alzheimer's disease, post-traumatic encephalomalacia, and headache. So they are recommended to be used directly for patient scans without acute pathology like significant midline shifts, bleeds, or mass effects. 

In [ ]:
ei_calc_processing_params = dict()
ei_calc_processing_params["ei_x_num_slices_above_ac_to_eval"] = 20
ei_calc_processing_params["around_lv_roi_col_left_prcntl_lim"] = 20
ei_calc_processing_params["around_lv_roi_col_right_prcntl_lim"] = 80
ei_calc_processing_params["around_lv_roi_row_upper_prcntl_lim"] = 5
ei_calc_processing_params["around_lv_roi_row_lower_prcntl_lim"] = 95
ei_calc_processing_params["around_lv_roi_disk_size"] = 10
ei_calc_processing_params["axial_adaptive_thresh_size"] = 105
ei_calc_processing_params["axial_adaptive_thresh_mean_thresh"] = 7.5
ei_calc_processing_params["otsu_adaptive_diff_thresh"] = 0.35
ei_calc_processing_params["otsu_adaptive_horizontal_thresh"] = 7
ei_calc_processing_params["lv_h_roi_col_thresh_prcnt"] = 0.075
ei_calc_processing_params["lv_h_roi_top_row_lim"] = 40
ei_calc_processing_params["lv_h_roi_bot_row_lim"] = 15

##### EI-x pipeline

In [ ]:
for acc_num in accnum_list:

    pat = info_df[pat_id_col_name][info_df[scan_id_col_name]==acc_num].values[0]
    
    #Gather AC-PC coordinates on the aligned axial volume
    pca_str = info_df[pc_coordinates_col_name][info_df[scan_id_col_name]==acc_num].values[0]
    [x,y,z] = pca_str.split(",")
    pca_x,pca_y,pca_z = [float(x.strip("[")),float(y),float(z.strip("]"))]      
    aca_str = info_df[ac_coordinates_col_name][info_df[scan_id_col_name]==acc_num].values[0]
    [x,y,z] = aca_str.split(",")
    aca_x,aca_y,aca_z = [float(x.strip("[")),float(y),float(z.strip("]"))]   
    ac = [aca_x,aca_y,aca_z]

  
    print(f'Patient is {pat} and acc_num is {acc_num}')

    study_path = os.path.join(data_path, acc_num, axial_acpc_vol_aligned_name)

    #Read in axial volume, fix rotation so it's displayed upright, and translate it so that the AC is at the origin
    axial_img = sitk.ReadImage(study_path)
    axial_img_brain = window_stack_sitk(rotate_image(axial_img))

    axial_img_brain_translated = translate_image(axial_img_brain, 3, ac)
    
    axial_data = sitk.GetArrayFromImage(axial_img_brain_translated) 
    N = axial_data.shape
    ac_indices = axial_img_brain_translated.TransformPhysicalPointToIndex([0,0,0])

    #Call function to evaluate EI-x
    ei, plane_opt = ei_calc(axial_data, ac_indices, ei_calc_processing_params, visualize = False)

    #Record calculated EI-x
    ei_x_df = pd.concat([ei_x_df, pd.DataFrame({pat_id_col_name:[pat], scan_id_col_name:["'" + acc_num + "'"],'ei_x':[ei]})],ignore_index = True)


In [ ]:
len(ei_x_df), sum(ei_x_df['ei_x'].isna())

In [ ]:
#Examine histogram of computed values
plt.hist(ei_x_df['ei_x'])
plt.xticks([x for x in np.arange(0, 0.7, 0.1)])
plt.show()

In [ ]:
#Check for any extreme values and inspect the corresponding outputs above for computational errors
min_ei_x_lim = 0.1
max_ei_x_lim = 0.6

ei_x_df[(ei_x_df['ei_x'] <= min_ei_x_lim) | (ei_x_df['ei_x'] >= max_ei_x_lim)]

In [ ]:
ei_x_df.to_csv(os.path.join(output_folder, 'ei_x.csv'))